### VT-NIRS: Virtual Twin for Non-Invasive Respiratory Support

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)   


### Library Loading

In [ ]:
import os, sys, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

import torch

from models.vt_nirs import VTNIRSModel
from models.baselines import CustomTLearner, CustomCausalForest, run_baselines
from utils.losses import CensoringAwareAdversarialLoss
from utils.metrics import (
    compute_all_metrics, c_for_benefit, plot_model_comparison_bars,
    plot_ite_distribution, plot_training_curves,
    plot_decomposed_ite_scatter, plot_subgroup_ite_trends
)
from utils.loader import COVARIATES_23, create_dataloaders, NIRSTwinDataset
from utils.extraction import (
    init_client, run_mimic_extraction, propensity_score_match,
    normalize_and_mask, standardize_features,
    extract_eicu_cohort, assign_eicu_treatment,
    extract_eicu_covariates, compute_eicu_vfd28,
    extract_eicu_temporal, build_eicu_temporal_sequences,
    propensity_score_match_baseline,
    FEATURE_COLS, TEMPORAL_FEATURE_COLS, N_TEMPORAL_COVARIATES,
)
from training.train import train_stage1, train_stage2, evaluate, DEFAULT_CONFIG

warnings.filterwarnings('ignore')
print('All imports successful.')
print(f'PyTorch: {torch.__version__} | Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

In [ ]:
BILLING_PROJECT = "YOUR_GCP_PROJECT_ID"  # <-- Set your Google Cloud billing project ID here

import utils.extraction as _ext
_ext.BILLING_PROJECT = BILLING_PROJECT
print(f'Billing project set to: {BILLING_PROJECT}')

In [ ]:
RANDOM_STATE   = 42
SEQUENCE_LEN   = 48
N_COVARIATES   = 23
D_MODEL        = 128
N_HEADS        = 4
N_LAYERS       = 4
D_FF           = 256
NOISE_DIM      = 8
HIDDEN_DIM     = 128
DROPOUT        = 0.1
BATCH_SIZE     = 128
EPOCHS_STAGE1  = 100
EPOCHS_STAGE2  = 50
LEARNING_RATE  = 2e-4
WEIGHT_DECAY   = 1e-4
PATIENCE       = 20
LAMBDA_ADV     = 1.0
LAMBDA_SURV    = 1.0
LAMBDA_VFD     = 0.5
LAMBDA_CONSIST = 0.5
LAMBDA_GATE    = 0.1
LAMBDA_IPM     = 1.0
LAMBDA_DR      = 2.0
LAMBDA_GP      = 10.0

OUTPUT_DIR = os.path.join(os.getcwd(), 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f'Config loaded. Device: {DEVICE}')


### MIMIC-IV Data Extraction

In [ ]:
print('Running MIMIC-IV BigQuery extraction...')
client = init_client()
data = run_mimic_extraction(client, seq_len=SEQUENCE_LEN)

X_all     = data['X']
W_all     = data['W']
VFD_all   = data['VFD']
D_all     = data['D']
valid_ids = data['valid_ids']
df_cohort = data['df_cohort']
df_vfd    = data['df_vfd']
df_cov    = data['df_cov']

X_baseline = df_cov[FEATURE_COLS].values
print(f'MIMIC-IV: X={X_all.shape}, N={len(df_cohort):,} patients')

### Propensity Score Matching & DataLoaders

In [ ]:
psm_idx, ps_model, ps = propensity_score_match(
    X_all, W_all, caliper_scale=0.2, random_state=RANDOM_STATE)

X_psm = X_all[psm_idx]
W_psm = W_all[psm_idx]
VFD_psm = VFD_all[psm_idx]
D_psm = D_all[psm_idx]

print(f'NIRS mean VFD-28: {VFD_psm[W_psm==1].mean():.2f}')
print(f'IMV  mean VFD-28: {VFD_psm[W_psm==0].mean():.2f}')

In [ ]:
X_scaled, pad_masks, ts_scaler = normalize_and_mask(X_psm, N_COVARIATES)

train_loader, val_loader, test_loader = create_dataloaders(
    sequences=X_scaled, treatments=W_psm,
    vfd_observed=VFD_psm, delta=D_psm,
    pad_masks=pad_masks, batch_size=BATCH_SIZE,
    val_fraction=0.15, test_fraction=0.15,
    random_state=RANDOM_STATE)

sample_batch = next(iter(train_loader))
for k, v in sample_batch.items():
    print(f'  {k}: {v.shape}')

### Model Initialization & Training

In [ ]:
model = VTNIRSModel(
    n_covariates=N_COVARIATES, d_model=D_MODEL, n_heads=N_HEADS,
    n_layers=N_LAYERS, d_ff=D_FF, noise_dim=NOISE_DIM,
    hidden_dim=HIDDEN_DIM, dropout=DROPOUT,
).to(DEVICE)

loss_fn = CensoringAwareAdversarialLoss(
    lambda_adv=LAMBDA_ADV, lambda_surv=LAMBDA_SURV,
    lambda_vfd=LAMBDA_VFD, lambda_consist=LAMBDA_CONSIST,
    lambda_gate=LAMBDA_GATE,
    lambda_ipm=LAMBDA_IPM,
    lambda_dr=LAMBDA_DR,
)

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')
print(f'  Propensity head params: {sum(p.numel() for p in model.propensity_head.parameters()):,}')
print(f'  Discriminator params: {sum(p.numel() for p in model.discriminator.parameters()):,}')

model.eval()
with torch.no_grad():
    pred_test, enc_out = model.forward_predictor(
        sample_batch['x'].to(DEVICE), sample_batch['pad_mask'].to(DEVICE))
print(f'Forward pass OK. ITE shape: {pred_test["ite"].shape}')
print(f'Propensity logits shape: {enc_out[5].shape}')

In [ ]:
config = DEFAULT_CONFIG.copy()
config.update({
    'n_covariates': N_COVARIATES, 'd_model': D_MODEL, 'n_heads': N_HEADS,
    'n_layers': N_LAYERS, 'd_ff': D_FF, 'noise_dim': NOISE_DIM,
    'hidden_dim': HIDDEN_DIM, 'dropout': DROPOUT,
    'epochs_stage1': EPOCHS_STAGE1, 'lr_generator': LEARNING_RATE,
    'lr_discriminator': LEARNING_RATE, 'lr_encoder': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY, 'batch_size': BATCH_SIZE,
    'device': str(DEVICE), 'save_dir': OUTPUT_DIR, 'patience': PATIENCE,
    'lambda_ipm': LAMBDA_IPM, 'lambda_dr': LAMBDA_DR, 'lambda_gp': LAMBDA_GP,
})

print(f'Stage 1: {EPOCHS_STAGE1} epochs, LR={LEARNING_RATE}, patience={PATIENCE}')
t0 = time.time()
model, log_stage1 = train_stage1(model, train_loader, val_loader, loss_fn, config, OUTPUT_DIR)
print(f'Stage 1 done in {(time.time()-t0)/60:.1f} min')

In [ ]:
config['epochs_stage2'] = EPOCHS_STAGE2
config['lr_predictor'] = LEARNING_RATE

t0 = time.time()
model, log_stage2 = train_stage2(model, train_loader, val_loader, loss_fn, config, OUTPUT_DIR)
print(f'Stage 2 done in {(time.time()-t0)/60:.1f} min')

### Evaluation & Visualization

In [ ]:
metrics, predictions = evaluate(model, test_loader, config)
print('\nVT-NIRS Test Results')
print('=' * 50)
for k, v in sorted(metrics.items()):
    if isinstance(v, float):
        print(f'  {k:30s}: {v:.4f}')
    else:
        print(f'  {k:30s}: {v}')

In [ ]:
if log_stage1 is not None:
    fig = plot_training_curves(log_stage1, save_path=f'{OUTPUT_DIR}/fig_training_curves.png')
    plt.show()
else:
    print('No training logs available.')

In [ ]:
ite_pred = predictions['ite'].squeeze()
fig = plot_ite_distribution(ite_pred, model_name='VT-NIRS',
                             save_path=f'{OUTPUT_DIR}/fig_ite_distribution.png')
plt.show()
print(f'Mean ITE: {ite_pred.mean():.4f} | NIRS beneficial: {(ite_pred>0).mean()*100:.1f}%')

In [ ]:
fig = plot_decomposed_ite_scatter(predictions, save_path=f'{OUTPUT_DIR}/fig_decomposed_ite.png')
plt.show()

In [ ]:
test_n = len(ite_pred)
test_idx = np.arange(len(X_scaled))[-test_n:]
X_test_baseline = X_scaled[test_idx, -1, :]

for feat_name, label in [('fio2_X', 'FiO2'), ('sofa_X', 'SOFA Score'), ('age_X', 'Age')]:
    idx = FEATURE_COLS.index(feat_name)
    fig = plot_subgroup_ite_trends(
        ite_pred, X_test_baseline[:, idx], f'{label} (baseline)',
        save_path=f'{OUTPUT_DIR}/fig_ite_trend_{feat_name}.png')
    plt.show()

### Baseline Comparisons (Custom T-Learner & Causal Forest)

In [ ]:
psm_stay_ids = [valid_ids[j] for j in psm_idx]
cov_indexed = df_cov.set_index('stay_id')
X_baseline_psm = cov_indexed.loc[psm_stay_ids, FEATURE_COLS].fillna(0).values

baseline_results = run_baselines(
    X_baseline_psm, W_psm, VFD_psm,
    n_estimators=500, random_state=RANDOM_STATE
)
print('Baseline models trained.')

### Ablation Study

In [ ]:
ABLATION_CKPT = f'{OUTPUT_DIR}/ablation_base/best_stage2.pth'

model_base = VTNIRSModel(
    n_covariates=N_COVARIATES, d_model=D_MODEL, n_heads=N_HEADS,
    n_layers=N_LAYERS, d_ff=D_FF, noise_dim=NOISE_DIM,
    hidden_dim=HIDDEN_DIM, dropout=DROPOUT,
).to(DEVICE)

loss_fn_base = CensoringAwareAdversarialLoss(
    lambda_adv=LAMBDA_ADV, lambda_surv=LAMBDA_SURV,
    lambda_vfd=LAMBDA_VFD, lambda_consist=LAMBDA_CONSIST,
    lambda_gate=0.0, lambda_ipm=LAMBDA_IPM, lambda_dr=LAMBDA_DR,
)

config_base = config.copy()
config_base['save_dir'] = f'{OUTPUT_DIR}/ablation_base'
os.makedirs(config_base['save_dir'], exist_ok=True)

if os.path.exists(ABLATION_CKPT):
    print(f'Loading ablation model from {ABLATION_CKPT}')
    model_base.load_state_dict(torch.load(ABLATION_CKPT, map_location=DEVICE))
    metrics_base, preds_base = evaluate(model_base, test_loader, config_base)
else:
    print('Training VT-NIRS-base (no gate entropy)...')
    torch.manual_seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(RANDOM_STATE)
    model_base, _ = train_stage1(model_base, train_loader, val_loader, loss_fn_base, config_base, config_base['save_dir'])
    model_base, _ = train_stage2(model_base, train_loader, val_loader, loss_fn_base, config_base, config_base['save_dir'])
    metrics_base, preds_base = evaluate(model_base, test_loader, config_base)
    torch.save(model_base.state_dict(), ABLATION_CKPT)

print('\nVT-NIRS-base results:')
for k, v in sorted(metrics_base.items()):
    if isinstance(v, float): print(f'  {k}: {v:.4f}')

### eICU External Validation

In [ ]:
print('Running eICU BigQuery extraction...')
df_eicu     = extract_eicu_cohort()
df_eicu_tx  = assign_eicu_treatment(df_eicu)
df_eicu_cov = extract_eicu_covariates(df_eicu_tx)
df_eicu_vfd = compute_eicu_vfd28(df_eicu_tx)
df_eicu_temporal = extract_eicu_temporal(df_eicu_tx)
X_eicu_ts, W_eicu_ts, VFD_eicu_ts, D_eicu_ts, eicu_valid_ids = (
    build_eicu_temporal_sequences(
        df_eicu_temporal, df_eicu_tx, df_eicu_vfd, df_eicu_cov,
        seq_len=SEQUENCE_LEN))

print(f'\neICU cohort: {len(df_eicu_tx):,} patients')
print(f'  NIRS: {(df_eicu_tx.Treatment_W==1).sum():,}')
print(f'  IMV:  {(df_eicu_tx.Treatment_W==0).sum():,}')
print(f'  VFD-28 mean: {df_eicu_vfd.vfd28.mean():.1f}')

if X_eicu_ts.shape[0] > 0:
    X_eicu_scaled, eicu_pad_masks, _ = normalize_and_mask(X_eicu_ts, N_TEMPORAL_COVARIATES)
    print(f'\neICU temporal ready: {X_eicu_scaled.shape}')
else:
    print('WARNING: No eICU temporal sequences built')

In [ ]:
df_eicu_cov_std, _ = standardize_features(df_eicu_cov, reference_df=df_cov)
eicu_tx_ids = set(df_eicu_vfd['stay_id'].values)
df_eicu_cov_tx = df_eicu_cov_std[df_eicu_cov_std['stay_id'].isin(eicu_tx_ids)].copy()
df_eicu_merged = df_eicu_cov_tx.merge(
    df_eicu_vfd[['stay_id', 'Treatment_W', 'vfd28']], on='stay_id', how='inner')

X_eicu_baseline = df_eicu_merged[FEATURE_COLS].fillna(0).values
W_eicu_bl = df_eicu_merged['Treatment_W'].values
VFD_eicu_bl = df_eicu_merged['vfd28'].values

print(f'eICU validation: {len(W_eicu_bl):,} patients')

eicu_psm_idx, _, _ = propensity_score_match_baseline(
    X_eicu_baseline, W_eicu_bl, FEATURE_COLS,
    caliper_scale=0.2, random_state=RANDOM_STATE)
X_eicu_psm = X_eicu_baseline[eicu_psm_idx]
W_eicu_psm = W_eicu_bl[eicu_psm_idx]
VFD_eicu_psm = VFD_eicu_bl[eicu_psm_idx]

print(f'\neICU post-PSM: NIRS VFD-28={VFD_eicu_psm[W_eicu_psm==1].mean():.2f}, '
      f'IMV VFD-28={VFD_eicu_psm[W_eicu_psm==0].mean():.2f}')

eicu_baseline_results = run_baselines(
    X_eicu_psm, W_eicu_psm, VFD_eicu_psm, random_state=RANDOM_STATE)

for name, res in eicu_baseline_results.items():
    ite = res['ite']
    print(f'  {name}: ATE={ite.mean():.4f}, std={ite.std():.4f}, '
          f'NIRS beneficial={(ite>0).mean()*100:.1f}%')

if X_eicu_ts.shape[0] > 0:
    print('\n' + '=' * 55)
    print('  VT-NIRS on eICU (transported validation)')
    print('=' * 55)
    from torch.utils.data import DataLoader
    eicu_dataset = NIRSTwinDataset(
        sequences=X_eicu_scaled, treatments=W_eicu_ts,
        vfd_observed=VFD_eicu_ts, delta=D_eicu_ts,
        pad_masks=eicu_pad_masks)
    eicu_loader = DataLoader(eicu_dataset, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=0, pin_memory=True)
    model.eval()
    eicu_vtnirs_metrics, eicu_vtnirs_preds = evaluate(model, eicu_loader, config)
    eicu_ite = eicu_vtnirs_preds['ite'].squeeze()

    print(f'\nVT-NIRS on eICU: ATE={eicu_ite.mean():.4f}, std={eicu_ite.std():.4f}, '
          f'NIRS beneficial={(eicu_ite>0).mean()*100:.1f}%')

    print('\n' + '=' * 55)
    print('  ALL METHODS COMPARISON')
    print('=' * 55)
    for name, tau in [('T-Learner', eicu_baseline_results['T-Learner']['ite']),
                      ('Causal Forest', eicu_baseline_results['Causal Forest']['ite']),
                      ('VT-NIRS (ours)', eicu_ite)]:
        print(f'  {name}: ATE={tau.mean():.4f}, NIRS beneficial={(tau>0).mean()*100:.1f}%')
else:
    print('\nSkipping VT-NIRS on eICU -- no temporal sequences available')

### Model Comparison

In [ ]:
test_n = len(ite_pred)
test_idx = np.arange(len(W_psm))[-test_n:]

tl_ite_test = baseline_results['T-Learner']['ite'][test_idx]
cf_ite_test = baseline_results['Causal Forest']['ite'][test_idx]

W_test = W_psm[test_idx]
VFD_test = VFD_psm[test_idx]

def policy_value(ite, W, VFD):
    recommend_nirs = (ite > 0).astype(float)
    match = (recommend_nirs == W).astype(float)
    return float(np.mean(match * VFD))

all_results = {
    'T-Learner': {'mean': policy_value(tl_ite_test, W_test, VFD_test), 'std': 0.0},
    'Causal Forest': {'mean': policy_value(cf_ite_test, W_test, VFD_test), 'std': 0.0},
    'VT-NIRS-base': {'mean': float(metrics_base.get('vfd_model_policy', 0)), 'std': 0.0},
    'VT-NIRS': {'mean': float(metrics.get('vfd_model_policy', 0)), 'std': 0.0},
}

fig = plot_model_comparison_bars(
    all_results, 'vfd_model_policy', 'Policy Value (VFD-28 days)',
    save_path=f'{OUTPUT_DIR}/fig_model_comparison.png')
plt.show()